# 5. Edges 总结


## 5.1. LangGraph 官方对边的定义和分类
`Edges(边)` 定义了节点之间的依赖关系，决定图如何根据当前状态决定下一步，以及何时停止运行。

边主要有以下几种类型：

### 5.1.1 普通边(Normal Edges)
**普通边** 表示从当前节点直接流转到下一个节点。

例如
```python
builder.add_edge("node_a", "node_b")
```
其含义是：node_a 执行完成后，下一步执行 node_b。

### 5.1.2 条件边(Conditional Edges)
**条件边** 表示当前节点执行完成后，调用一个 路由函数 来决定下一步进入哪个或哪些节点。

例如
```python
builder.add_conditional_edges("node_a", router)
```
其中，`router` 会根据当前图状态返回后续目标节点。

如果**路由函数**返回一个节点名，则下一步进入该节点；如果返回多个节点名，则这些目标节点会在下一个超步中并行执行。

### 5.1.3 入口点(Entry Point)
**入口点** 表示图开始运行时，首先执行哪个节点。

本质上，入口点就是一条 以 START 为起点的普通边。

例如
```python
builder.add_edge(START, "node_a")
```
其含义是：当用户输入进入图时，首先执行 `node_a`。

LangGraph 也提供了专门的 API 来声明入口点：

```python
builder.set_entry_point("node_a")
```
二者是等价的。

### 5.1.4 条件入口点(Conditional Entry Point)
**条件入口点** 表示图开始运行时，并不固定进入某一个节点，而是先调用一个 **路由函数**，再根据路由结果决定首先执行哪个节点，或者哪些节点。

本质上，条件入口点就是一条 以 `START` 为起点的条件边。

例如
```python
builder.add_conditional_edges(START, router)
```
其含义是：图开始运行时，先执行 router，再根据 router 的返回值决定首个执行节点。

### 5.1.5 注意
1. 一个节点可以有多个出边。此时，这些出边对应的目标节点会在下一个超步中并行执行。
2. 除了上述方式，条件边还可以通过在节点中返回 `Command(goto=...)` 来实现，虽然方式不同，但边的性质和作用完全一致。
3. 在实际开发中，建议同一个节点尽量只选择一种路由机制：
    - 要么使用普通边，固定流转；
    - 要么使用条件边或 `Command(goto=...)`，动态流转。

不要在同一个节点上同时混用普通边和动态路由，否则多个路径可能同时生效，最终导致运行图逻辑混乱，与预期不符。

## 5.2. 指向END的终止边
除了官方文档中列出的几种边，还有一类非常重要的边：指向 `END` 的边。

严格来说，`END` 不是普通业务节点，而是 LangGraph 中的特殊终止标记。当流程流转到 `END` 时，表示该路径不再继续执行后续节点。

为了便于理解，可以将指向 `END` 的边分为两种情况。

### 5.2.1 终止点（Finish Point）
**终止点** 表示某个节点执行完成后，图流程结束。

本质上，它是一条 以 `END` 为目标点的普通边。

例如
```python
builder.add_edge("node_a", END)
```
其含义是：当 node_a 执行完成后，当前路径终止，不再继续执行其他节点。

LangGraph 也提供了专门的 API 来声明结束点：

```python
builder.set_finish_point("node_a")
```
二者是等价的。

### 5.2.2 条件终止点（Conditional Finish Point）
**条件终止点** 表示图在运行过程中，通过 **路由函数** 判断是继续执行后续节点，还是终止当前路径。

本质上，它是一条 路由结果中包含 `END` 的条件边。

例如
```python
def router(state: EmptyState) -> Literal["node_b", END]:
    if state["is_finished"]:
        return END
    return "node_b"

builder.add_conditional_edges("node_a", router)
```

其含义是：

- 如果 `state["is_finished"]` 为真，则返回 `END`，流程终止；
- 否则返回 `"node_b"`，继续执行 `node_b`。


## 5.3 `END`和终止边的说明
在编译后的运行图中，终止边 和 `END` 并没有具体的 `Channel` 和 节点 与之对应，它们只是终止标记，表示运行图的结束。
终止边的声明可以省略，LangGraph 运行时会在没有活跃节点时自然终止。但是，为了程序的可读性和图结构可视化渲染器的正常工作，最好显式声明指向 `END` 的边。